In [83]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

In [2]:
train=pd.read_csv("../data/raw/Train.csv")
test=pd.read_csv("../data/raw/Test.csv")

In [3]:
train["time"]=pd.to_datetime(train["time"])
test["time"]=pd.to_datetime(test["time"])

In [4]:
train["month"]=train["time"].dt.month
test["month"]=test["time"].dt.month

In [5]:
train["month_sin"]=np.sin(2 * np.pi * train["month"]/12)
train["month_cos"]=np.cos(2 * np.pi * train["month"]/12)
test["month_sin"]=np.sin(2 * np.pi * test["month"]/12)
test["month_cos"]=np.cos(2 * np.pi * test["month"]/12 )

In [6]:
X=train.drop(columns=["sample_id","target","time"])
y=train["target"]

In [87]:
X_train,X_val,y_train,y_val=train_test_split(X,y,random_state=42,test_size=0.2)

In [88]:
model=CatBoostRegressor(random_state=42,learning_rate=0.15,iterations=800,verbose=0,depth=10)

In [89]:
model.fit(X_train,y_train)

CatBoostRegressor(depth=10, iterations=800, learning_rate=0.15, loss_function='RMSE', random_state=42, verbose=0)

In [90]:
y_pred=model.predict(X_val)

In [91]:
rmse = root_mean_squared_error(y_val, y_pred)

print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 0.4468


In [92]:
print(X.columns.tolist())

['lat', 'lon', 'TWS_t', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'month_sin', 'month_cos', 'month']


In [93]:
print(train.columns.tolist())

['sample_id', 'time', 'lat', 'lon', 'TWS_t', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'month_sin', 'month_cos', 'target', 'month']


In [94]:
print(model.get_params())

{'iterations': 800, 'learning_rate': 0.15, 'depth': 10, 'loss_function': 'RMSE', 'verbose': 0, 'random_state': 42}


In [95]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [96]:
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=800,
        learning_rate=0.15,
        depth=10,
        random_state=42,
        verbose=0
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    rmse = root_mean_squared_error(y_val, preds)

    scores.append(rmse)

    print(f"Fold {fold}: {rmse:.4f}")

Fold 1: 0.4462
Fold 2: 0.4461
Fold 3: 0.4461
Fold 4: 0.4460
Fold 5: 0.4468


In [97]:
print(f"\nAverage RMSE: {np.mean(scores):.4f}")
print(f"Std RMSE: {np.std(scores):.4f}")


Average RMSE: 0.4462
Std RMSE: 0.0003


In [98]:
best_model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.15,
    depth=10,
    random_state=42,
    verbose=0
)

best_model.fit(X, y)

CatBoostRegressor(depth=10, iterations=800, learning_rate=0.15, loss_function='RMSE', random_state=42, verbose=0)

In [100]:
submission = pd.read_csv("../data/raw/SampleSubmission.csv")

In [101]:
X_test = test[X.columns]

In [102]:
predictions = best_model.predict(X_test)
submission["Target"] = predictions
submission.to_csv("submission_catboost.csv", index=False)


In [103]:
submission.head()

,ID,Target
0,20150901_-55.5_-68.5,0.312371
1,20150901_-55.5_-67.5,0.440431
2,20150901_-54.5_-71.5,0.558096
3,20150901_-54.5_-70.5,0.416382
4,20150901_-54.5_-69.5,0.435712
